This is the notebook for cleaning and wrangling the wetlands shapefile. Steps include:
1. Adding the Hectares column and subsetting the data to polygons larger than 1 Hectare.
2. Removing shapefile length, shapefile area, and QAQC. 

In [1]:
# import packages
import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd

Access Initial Shapefile from Box Drive and read in the shapefile

In [2]:
# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
# uploading wetlands that are bigger than a hectare
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands' / 'wetlands.shp'
print(wetlands_path.exists())



Box folder found at /Users/masonbunker/Library/CloudStorage/Box-Box
True


In [3]:
#read in shp, be PATIENT (took MB 4.5 mins)
wetlands_shp = gpd.read_file(wetlands_path)

Find out information from the wetlands shapefile

In [4]:
wetlands_shp.shape
wetlands_shp.columns


Index(['OBJECTID', 'ATTRIBUTE', 'WETLAND_TY', 'QAQC_CODE', 'ACRES', 'NWI_ID',
       'Shape_Leng', 'Shape_Area', 'geometry'],
      dtype='str')

In [5]:
wetlands_shp.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 891086 entries, 0 to 891085
Data columns (total 9 columns):
 #   Column      Non-Null Count   Dtype   
---  ------      --------------   -----   
 0   OBJECTID    891086 non-null  float64 
 1   ATTRIBUTE   891086 non-null  str     
 2   WETLAND_TY  891086 non-null  str     
 3   QAQC_CODE   22260 non-null   str     
 4   ACRES       891086 non-null  float64 
 5   NWI_ID      891086 non-null  str     
 6   Shape_Leng  891086 non-null  float64 
 7   Shape_Area  891086 non-null  float64 
 8   geometry    891086 non-null  geometry
dtypes: float64(4), geometry(1), str(4)
memory usage: 61.2 MB


Subset the data to only include polygons larger than 1 Hectare (10,000m^2), roughly 20% of the initial data set.

In [6]:
wetlands_shp["Hectares"] = wetlands_shp["Shape_Area"]/10000

wetlands_hectare = wetlands_shp.loc[lambda df: df["Hectares"] > 1]

wetlands_hectare.shape

#calculate subset percent
subset = 179139/891086*100
print(subset)

20.103446805358853


Determine appropriate columns to remain in the cleaned dataset.

In [7]:
wetlands_hectare.head()

,OBJECTID,ATTRIBUTE,WETLAND_TY,QAQC_CODE,ACRES,NWI_ID,Shape_Leng,Shape_Area,geometry,Hectares
2,3.0,R4SBC,Riverine,NaN,9.394554,202409CSw{B818AEF7-DD4D-43B6-A532-1DC94D248F88...,12669.925427,38018.410790,"POLYGON ((-2195423.854 1489042.416, -2195423.8...",3.801841
9,10.0,R4SBC,Riverine,NaN,4.869321,202409CSw{80E28DA5-6244-45AB-9DC9-774ACABD2846...,6572.482823,19705.442594,"POLYGON ((-2197428.286 1494975.476, -2197428.3...",1.970544
11,12.0,R4SBC,Riverine,NaN,7.364585,202409CSw{A66FA93B-E12A-488A-9891-E7526D1AACCE...,9937.020983,29803.416122,"POLYGON ((-2193923.26 1496246.62, -2193923.139...",2.980342
12,13.0,R4SBC,Riverine,NaN,3.099673,202409CSw{B5A926CC-AF98-487F-A4B9-A4789311100A...,4194.847948,12543.932403,"POLYGON ((-2194477.48 1496368.039, -2194475.19...",1.254393
14,15.0,R4SBC,Riverine,NaN,20.027313,202409CSw{C99C89FF-6622-43DE-A93F-138503071611...,26989.063820,81047.662073,"POLYGON ((-2195115.466 1496641.214, -2195115.3...",8.104766


Subset to only Core Columns and rename to be more appropriate

In [20]:
core_cols = ["ATTRIBUTE", "WETLAND_TY","Hectares", "NWI_ID", "geometry"]

wetlands = wetlands_hectare[core_cols]

In [21]:
wetlands = wetlands.rename(
    columns={
        "ATTRIBUTE" : "C Code",
        "WETLAND_TY" : "NWI Type",
        "NWI_ID": "NWI ID"
    }
)

wetlands.head()
#wetlands.shape

,C Code,NWI Type,Hectares,NWI ID,geometry
2,R4SBC,Riverine,3.801841,202409CSw{B818AEF7-DD4D-43B6-A532-1DC94D248F88...,"POLYGON ((-2195423.854 1489042.416, -2195423.8..."
9,R4SBC,Riverine,1.970544,202409CSw{80E28DA5-6244-45AB-9DC9-774ACABD2846...,"POLYGON ((-2197428.286 1494975.476, -2197428.3..."
11,R4SBC,Riverine,2.980342,202409CSw{A66FA93B-E12A-488A-9891-E7526D1AACCE...,"POLYGON ((-2193923.26 1496246.62, -2193923.139..."
12,R4SBC,Riverine,1.254393,202409CSw{B5A926CC-AF98-487F-A4B9-A4789311100A...,"POLYGON ((-2194477.48 1496368.039, -2194475.19..."
14,R4SBC,Riverine,8.104766,202409CSw{C99C89FF-6622-43DE-A93F-138503071611...,"POLYGON ((-2195115.466 1496641.214, -2195115.3..."


Write a function to fill in the columns for Cowardin (C) Subsystem and Class

In [22]:
# create additional columns for subsystem and class
wetlands.insert(2, "C System", None)
wetlands.insert(3, "C Subsystem", None)
wetlands.insert(4, "C Class", None)

wetlands.head()

,C Code,NWI Type,C System,C Subsystem,C Class,Hectares,NWI ID,geometry
2,R4SBC,Riverine,None,None,None,3.801841,202409CSw{B818AEF7-DD4D-43B6-A532-1DC94D248F88...,"POLYGON ((-2195423.854 1489042.416, -2195423.8..."
9,R4SBC,Riverine,None,None,None,1.970544,202409CSw{80E28DA5-6244-45AB-9DC9-774ACABD2846...,"POLYGON ((-2197428.286 1494975.476, -2197428.3..."
11,R4SBC,Riverine,None,None,None,2.980342,202409CSw{A66FA93B-E12A-488A-9891-E7526D1AACCE...,"POLYGON ((-2193923.26 1496246.62, -2193923.139..."
12,R4SBC,Riverine,None,None,None,1.254393,202409CSw{B5A926CC-AF98-487F-A4B9-A4789311100A...,"POLYGON ((-2194477.48 1496368.039, -2194475.19..."
14,R4SBC,Riverine,None,None,None,8.104766,202409CSw{C99C89FF-6622-43DE-A93F-138503071611...,"POLYGON ((-2195115.466 1496641.214, -2195115.3..."


In [ ]:
# generate function to determine the Systems
def assign_system(df):
    wetlands.loc[lambda df: df["C Code"].str.startswith("M", na = False), "C System"] = 'Marine'
    wetlands.loc[lambda df: df["C Code"].str.startswith("E", na = False), "C System"] = 'Estuarine'
    wetlands.loc[lambda df: df["C Code"].str.startswith("R", na = False), "C System"] = 'Riverine'
    wetlands.loc[lambda df: df["C Code"].str.startswith("L", na = False), "C System"] = 'Lacustrine'
    wetlands.loc[lambda df: df["C Code"].str.startswith("P", na = False), "C System"] = 'Palustrine'

    return df

#apply system function
wetlands = assign_system(wetlands)
wetlands.sample(5)


,C Code,NWI Type,C System,C Subsystem,C Class,Hectares,NWI ID,geometry
127728,R4SBC,Riverine,Riverine,None,None,1.705808,202409CSw{111A7731-4BD6-4300-85FA-38557EA2C7E0...,"POLYGON ((-2165828.181 1742384.574, -2165829.9..."
275701,R4SBJ,Riverine,Riverine,None,None,1.023593,202409CSw{959BF5C9-E002-4307-A67B-DAA5B48F02B2...,"POLYGON ((-1733646.432 1308982.183, -1733658.6..."
403215,PEM1C,Freshwater Emergent Wetland,Palustrine,None,None,1.773527,202409CSw{1DA9FBC8-2D17-4A6A-8A56-1E963CF8A3CE...,"POLYGON ((-2253844.558 2126673.824, -2253837.4..."
847147,R4SBC,Riverine,Riverine,None,None,6.849028,202409CSw{2F831EDA-0867-48CE-BA8A-1B6421A4F197...,"POLYGON ((-2145093.892 2224025.427, -2145087.9..."
479829,R4SBJ,Riverine,Riverine,None,None,1.704713,202409CSw{8F6968DF-F438-4C92-A7F5-1BABFA6E7433...,"POLYGON ((-2240995.598 2091803.032, -2240999.9..."


In [37]:
# generate function to determine the Subsystems
def assign_subsystem(df):
    # marine
    wetlands.loc[lambda df: df["C Code"].str.startswith("M1", na = False), "C Subsystem"] = 'Subtidal'
    wetlands.loc[lambda df: df["C Code"].str.startswith("M2", na = False), "C Subsystem"] = 'Intertidal' 
    # estuarine
    wetlands.loc[lambda df: df["C Code"].str.startswith("E1", na = False), "C Subsystem"] = 'Subtidal'
    wetlands.loc[lambda df: df["C Code"].str.startswith("E2", na = False), "C Subsystem"] = 'Intertidal' 
    # riverine
    wetlands.loc[lambda df: df["C Code"].str.startswith("R1", na = False), "C Subsystem"] = 'Tidal'
    wetlands.loc[lambda df: df["C Code"].str.startswith("R2", na = False), "C Subsystem"] = 'Lower Perennial'
    wetlands.loc[lambda df: df["C Code"].str.startswith("R3", na = False), "C Subsystem"] = 'Upper Perennial'
    wetlands.loc[lambda df: df["C Code"].str.startswith("R4", na = False), "C Subsystem"] = 'Intermittent'
    wetlands.loc[lambda df: df["C Code"].str.startswith("R5", na = False), "C Subsystem"] = 'Unknown Perennial'

    # lacustrine
    wetlands.loc[lambda df: df["C Code"].str.startswith("L1", na = False), "C Subsystem"] = 'Limnetic'
    wetlands.loc[lambda df: df["C Code"].str.startswith("L2", na = False), "C Subsystem"] = 'Littoral'
    # palustrine
    wetlands.loc[lambda df: df["C Code"].str.startswith("P", na = False), "C Subsystem"] = "N/A"
    
    return df

## apply the subsystem function
wetlands = assign_subsystem(wetlands)

wetlands.head(12)

,C Code,NWI Type,C System,C Subsystem,C Class,Hectares,NWI ID,geometry
2,R4SBC,Riverine,Riverine,Intermittent,Streambed,3.801841,202409CSw{B818AEF7-DD4D-43B6-A532-1DC94D248F88...,"POLYGON ((-2195423.854 1489042.416, -2195423.8..."
9,R4SBC,Riverine,Riverine,Intermittent,Streambed,1.970544,202409CSw{80E28DA5-6244-45AB-9DC9-774ACABD2846...,"POLYGON ((-2197428.286 1494975.476, -2197428.3..."
11,R4SBC,Riverine,Riverine,Intermittent,Streambed,2.980342,202409CSw{A66FA93B-E12A-488A-9891-E7526D1AACCE...,"POLYGON ((-2193923.26 1496246.62, -2193923.139..."
12,R4SBC,Riverine,Riverine,Intermittent,Streambed,1.254393,202409CSw{B5A926CC-AF98-487F-A4B9-A4789311100A...,"POLYGON ((-2194477.48 1496368.039, -2194475.19..."
14,R4SBC,Riverine,Riverine,Intermittent,Streambed,8.104766,202409CSw{C99C89FF-6622-43DE-A93F-138503071611...,"POLYGON ((-2195115.466 1496641.214, -2195115.3..."
18,R4SBC,Riverine,Riverine,Intermittent,Streambed,6.543216,202409CSw{2748E15A-8C19-45BE-B3AE-EF4D3462394D...,"POLYGON ((-2204235.899 1502866.581, -2204233.8..."
19,R4SBC,Riverine,Riverine,Intermittent,Streambed,2.289157,202409CSw{A9FC265B-A070-49F6-91EB-1AF1B24F270E...,"POLYGON ((-2212223.548 1504505.924, -2212221.2..."
21,R4SBC,Riverine,Riverine,Intermittent,Streambed,1.366831,202409CSw{02608E9D-9216-4FBD-802A-AAD2E4A3047C...,"POLYGON ((-2211172.156 1506003.818, -2211172.0..."
34,R5UBF,Riverine,Riverine,Unknown Perennial,None,3.483020,202409CSw{731BDF60-58CE-48ED-9DB8-7558C8FC3E8E...,"POLYGON ((-2193015.598 1494952.847, -2193014.2..."
35,R4SBC,Riverine,Riverine,Intermittent,Streambed,1.250247,202409CSw{A80FA3E0-6491-4C92-95B6-4AB6D195B8B2...,"POLYGON ((-2195615.688 1490047.353, -2195617.4..."


In [43]:
#generate function to determine the Classes
def assign_class(df):
    wetlands.loc[lambda df: df["C Code"].str.contains("SB", na = False), "C Class"] = "Streambed"
    wetlands.loc[lambda df: df["C Code"].str.contains("RB", na = False), "C Class"] = "Rock Bottom"
    wetlands.loc[lambda df: df["C Code"].str.contains("UB", na = False), "C Class"] = "Unconsolidated Bottom"
    wetlands.loc[lambda df: df["C Code"].str.contains("AB", na = False), "C Class"] = "Aquatic Bed"
    wetlands.loc[lambda df: df["C Code"].str.contains("RF", na = False), "C Class"] = "Reef"
    wetlands.loc[lambda df: df["C Code"].str.contains("RS", na = False), "C Class"] = "Rocky Shore"
    wetlands.loc[lambda df: df["C Code"].str.contains("US", na = False), "C Class"] = "Unconsolidated Shore"
    wetlands.loc[lambda df: df["C Code"].str.contains("EM", na = False), "C Class"] = "Emergent"
    wetlands.loc[lambda df: df["C Code"].str.contains("SS", na = False), "C Class"] = "Scrub-Shrub"
    wetlands.loc[lambda df: df["C Code"].str.contains("FO", na = False), "C Class"] = "Forested"
    wetlands.loc[lambda df: df["C Code"].str.contains("ML", na = False), "C Class"] = "Moss-Lichen"
    
    return df

wetlands = assign_class(wetlands)

wetlands.sample(12)


,C Code,NWI Type,C System,C Subsystem,C Class,Hectares,NWI ID,geometry
846432,R3UBG,Riverine,Riverine,Upper Perennial,Unconsolidated Bottom,15.504892,202409CSw{6E9BB55D-2E72-481F-8EE9-BF30CB67A757...,"POLYGON ((-2116862.426 2287576.445, -2116867.5..."
419665,PUBHx,Freshwater Pond,Palustrine,N/A,Unconsolidated Bottom,3.145713,202409CSw{2EB6A27D-8896-457E-9B68-D205579C182C...,"POLYGON ((-2202715.107 2103920.009, -2202718.1..."
389420,R4SBJ,Riverine,Riverine,Intermittent,Streambed,1.363508,202512CSw{8A7C3A4C-3626-4264-BE2E-14F363A66F9D...,"POLYGON ((-1963396.917 1832583.551, -1963397.4..."
19924,PEM1A,Freshwater Emergent Wetland,Palustrine,N/A,Emergent,3.753455,202409CSw{3D84E12C-A204-40CA-AC99-8A71E33C517F...,"POLYGON ((-2207153.377 1607020.48, -2207171.89..."
597004,PEM1C,Freshwater Emergent Wetland,Palustrine,N/A,Emergent,2.135938,202409CSw{E5D37B19-A423-474B-8F87-2F8A54C08234...,"POLYGON ((-2027026.956 1888298.866, -2027026.9..."
296480,R4SBA,Riverine,Riverine,Intermittent,Streambed,1.312837,202409CSw{0D05EC55-C731-4E78-ADCE-540EA38573D0...,"POLYGON ((-1663050.882 1394603.479, -1663053.0..."
828901,PSS1A,Freshwater Forested/Shrub Wetland,Palustrine,N/A,Scrub-Shrub,3.342278,202409CSw{3CFCF09F-9B4F-4FFB-A865-739B598C17C8...,"POLYGON ((-2180166.601 2353944.489, -2180166.6..."
321790,R4SBJ,Riverine,Riverine,Intermittent,Streambed,4.947189,202409CSw{6CBCFFBF-5791-4FFE-AE0B-D76E6AEBA15F...,"POLYGON ((-1848490.25 1609918.34, -1848490.988..."
518507,R3UBH,Riverine,Riverine,Upper Perennial,Unconsolidated Bottom,3.884923,202409CSw{3D4FF5CB-8A48-4450-91BD-70ECC7AD0DFD...,"POLYGON ((-2110073.023 1951336.337, -2110072.9..."
316160,PSS1/EM1B,Freshwater Forested/Shrub Wetland,Palustrine,N/A,Scrub-Shrub,3.168614,202409CSw{7DFF2688-751C-46BA-8D76-E8D36067F315...,"POLYGON ((-1998033.742 1609558.421, -1998033.7..."


Export new shapefile  into box

In [76]:
# confirm box path still works

# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
wetlands_subset_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset'

print(wetlands_subset_gpkg_path.exists())

Box folder found at /Users/masonbunker/Library/CloudStorage/Box-Box
True


In [ ]:
# actually save to the the folder

# THIS STEP IS CAUSING PROBLEMS AND I CANT SOLVE IT
wetlands.to_file(wetlands_subset_path/"wetlands_subset.gpkg", driver="GPKG")

DataSourceError: sqlite3_open(/Users/masonbunker/Library/CloudStorage/Box-Box/ENVIRON 712L-Students/Data/BunkerAndersonTseng/wetlands_subset/wetlands_subset.gpkg/wetlands_subset.gpkg) failed: unable to open database file

Confirm that the export worked and the file can be read in.

In [79]:
# create path object that goes to the specific folder on Box drive
wetlands_subset_gpkg_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.gpkg'
print("It is", wetlands_subset_gpkg_path.exists(), "that the file exists.")

It is True that the file exists.


In [80]:
# read in shp, be PATIENT
wetlands_subset = gpd.read_file(wetlands_subset_gpkg_path)

DataSourceError: sqlite3_prepare_v2(SELECT COUNT(*) FROM sqlite_master WHERE name IN ('gpkg_metadata', 'gpkg_metadata_reference') AND type IN ('table', 'view')) failed: attempt to write a readonly database; attempt to write a readonly database

In [81]:
import os

print("File size (GB):",
      wetlands_subset_gpkg_path.stat().st_size / (1024**3))

print("Readable:",
      os.access(wetlands_subset_gpkg_path, os.R_OK))

File size (GB): 2.744964599609375
Readable: True


In [82]:
wetlands_subset = gpd.read_file(
    wetlands_subset_gpkg_path,
    engine="pyogrio"
)

DataSourceError: sqlite3_prepare_v2(SELECT COUNT(*) FROM sqlite_master WHERE name IN ('gpkg_metadata', 'gpkg_metadata_reference') AND type IN ('table', 'view')) failed: attempt to write a readonly database; attempt to write a readonly database